In [13]:
pip install scikit-learn imbalanced-learn joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [30]:
df= pd.read_csv("PS_20174392719_1491204439457_log.csv")

Importing the dataset

Checking the dataset first 5 rows and columns with all Values

In [31]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [32]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
step,6362620.0,2.433972e+02,1.423320e+02,1.0,156.00,239.000,3.350000e+02,7.430000e+02
amount,6362620.0,1.798619e+05,6.038582e+05,0.0,13389.57,74871.940,2.087215e+05,9.244552e+07
oldbalanceOrg,6362620.0,8.338831e+05,2.888243e+06,0.0,0.00,14208.000,1.073152e+05,5.958504e+07
newbalanceOrig,6362620.0,8.551137e+05,2.924049e+06,0.0,0.00,0.000,1.442584e+05,4.958504e+07
oldbalanceDest,6362620.0,1.100702e+06,3.399180e+06,0.0,0.00,132705.665,9.430367e+05,3.560159e+08
newbalanceDest,6362620.0,1.224996e+06,3.674129e+06,0.0,0.00,214661.440,1.111909e+06,3.561793e+08
isFraud,6362620.0,1.290820e-03,3.590480e-02,0.0,0.00,0.000,0.000000e+00,1.000000e+00
isFlaggedFraud,6362620.0,2.514687e-06,1.585775e-03,0.0,0.00,0.000,0.000000e+00,1.000000e+00


Duplicate Values Checking

In [33]:
df.duplicated().sum()


np.int64(0)

Checking for Duplicate Values

In [34]:
df.drop_duplicates(inplace=True)

Checking for the Missing Values

In [35]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Checking the number of Rows and Columns

In [36]:
df.shape

(6362620, 11)

Checking for Unique Values so that we can remove them as they don't contribute a lot to our dataset

In [37]:
df.nunique()

step                  743
type                    5
amount            5316900
nameOrig          6353307
oldbalanceOrg     1845844
newbalanceOrig    2682586
nameDest          2722362
oldbalanceDest    3614697
newbalanceDest    3555499
isFraud                 2
isFlaggedFraud          2
dtype: int64

Removing the unique values

In [38]:
df.drop(['nameOrig'], axis=1).head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,41554.0,29885.86,M1230701703,0.0,0.0,0,0


To check the edge / number of fraud and legit transactions

In [39]:
df['isFraud'].value_counts(normalize=False)

isFraud
0    6354407
1       8213
Name: count, dtype: int64

Counts the number of unique values of a particular column here it is "type" which is Transaction Type

In [40]:
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Categorising data based on Types which will help in reducing the data and classifying theminto Categories

In [41]:
df['type'] = df['type'].astype('category')

Here it is converted into an Object

In [42]:
df['type'].head()

0     PAYMENT
1     PAYMENT
2    TRANSFER
3    CASH_OUT
4     PAYMENT
Name: type, dtype: category
Categories (5, object): ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']

To reduce the memory space for the numerical values making preprocessing lighter and fast also useful during model training since we have 6 millions transactions data.

In [43]:
df['step'] = pd.to_numeric(df['step'], downcast='integer')
for col in ['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']:
    df[col] = pd.to_numeric(df[col], downcast='float')

This checks for any amount if it is negatuve i.e. less than zero which is not possible for any transaction to be negative.

In [44]:
df = df[df['amount'] >= 0]

To replace any values which are either +infinity or -infinity  to NA so it will be easy for models to get trained with as they cannot handle the infinity values.

In [45]:
df.replace([np.inf, -np.inf], pd.NA, inplace=True)

This step converts the "steps" in date, time, weeks so it will be easy to detect the fraud and also classify them.

In [46]:
base = pd.to_datetime("2017-01-01")   # arbitrary start
df['datetime'] = base + pd.to_timedelta(df['step'], unit='h')
df['hour'] = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek    # 0=Mon .. 6=Sun
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)


In [47]:
df.tail()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,datetime,hour,dayofweek,is_weekend
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0,2017-01-31 23:00:00,23,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0,2017-01-31 23:00:00,23,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0,2017-01-31 23:00:00,23,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0,2017-01-31 23:00:00,23,1,0
6362619,743,CASH_OUT,850002.52,C1280323807,850002.52,0.0,C873221189,6510099.11,7360101.63,1,0,2017-01-31 23:00:00,23,1,0


This check for legitmate tansactions amount that were transferred and after that what new amount has become for both sender and receiver.

In [48]:
df['errorBalanceOrig'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
df['has_balance_mismatch'] = ((df['errorBalanceOrig'].abs() > 1e-6) | 
                              (df['errorBalanceDest'].abs() > 1e-6)).astype(int)


In [49]:
df.tail()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,datetime,hour,dayofweek,is_weekend,errorBalanceOrig,errorBalanceDest,has_balance_mismatch
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0,2017-01-31 23:00:00,23,1,0,0.0,0.000000e+00,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0,2017-01-31 23:00:00,23,1,0,0.0,6.311409e+06,1
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0,2017-01-31 23:00:00,23,1,0,0.0,1.000000e-02,1
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0,2017-01-31 23:00:00,23,1,0,0.0,8.500025e+05,1
6362619,743,CASH_OUT,850002.52,C1280323807,850002.52,0.0,C873221189,6510099.11,7360101.63,1,0,2017-01-31 23:00:00,23,1,0,0.0,9.313226e-10,0


This reduces noise and dimensionaltiy making the dataset focused, lighter and effective by grouping the transaction types together for more focused UPI approach

In [50]:
mapping = {
    'CASH-IN': 'other',     # maybe ignore or keep as other
    'CASH-OUT': 'upi_transfer',
    'TRANSFER': 'upi_transfer',
    'PAYMENT': 'upi_payment',
    'DEBIT': 'other'
}
df['upi_type'] = df['type'].map(mapping).astype('category')


Grouping the Users based on their statistical transactions and reports in order to classify for legit and fraud users.

In [51]:
agg_orig = df.groupby('nameOrig')['amount'].agg(['count','sum','mean','max','std']).reset_index()
agg_orig.columns = ['nameOrig','orig_tx_count','orig_amount_sum','orig_amount_mean','orig_amount_max','orig_amount_std']
df = df.merge(agg_orig, on='nameOrig', how='left')

In [52]:
df.tail()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,is_weekend,errorBalanceOrig,errorBalanceDest,has_balance_mismatch,upi_type,orig_tx_count,orig_amount_sum,orig_amount_mean,orig_amount_max,orig_amount_std
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,...,0,0.0,0.000000e+00,0,NaN,1,339682.13,339682.13,339682.13,NaN
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,...,0,0.0,6.311409e+06,1,upi_transfer,1,6311409.28,6311409.28,6311409.28,NaN
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,...,0,0.0,1.000000e-02,1,NaN,1,6311409.28,6311409.28,6311409.28,NaN
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,...,0,0.0,8.500025e+05,1,upi_transfer,1,850002.52,850002.52,850002.52,NaN
6362619,743,CASH_OUT,850002.52,C1280323807,850002.52,0.0,C873221189,6510099.11,7360101.63,1,...,0,0.0,9.313226e-10,0,NaN,1,850002.52,850002.52,850002.52,NaN


Deleting the aggregations for leakge prevention into the training and testing datset

In [53]:
leaky_prefixes = ['orig_tx_count','orig_amount_sum','orig_amount_mean','orig_amount_max',
                   'dest_tx_count','dest_amount_sum','dest_amount_mean','dest_amount_max']
for col in list(df.columns):
    for p in leaky_prefixes:
        if col.startswith(p):
            df.drop(columns=[col], inplace=True)


Deleting all the values which have null for isFraud as they are not required for the model training.

In [54]:
if df['isFraud'].isnull().any():
    print("Dropping", df['isFraud'].isnull().sum(), "rows missing isFraud")
    df = df.dropna(subset=['isFraud'])

Analyzing for missing vakues across whole dataset.

In [55]:
missing_frac = df.isnull().mean().sort_values(ascending=False)
print("Columns with >0 missing fraction:\n", missing_frac[missing_frac > 0])

Columns with >0 missing fraction:
 orig_amount_std    0.997075
upi_type           0.571586
dtype: float64


Deleting columns with mising vakues greater then 50%.

In [56]:
high_missing = missing_frac[missing_frac > 0.5]
if len(high_missing) > 0:
    print("WARNING: columns with >50% missing (consider dropping or investigating):\n", high_missing)

 orig_amount_std    0.997075
upi_type           0.571586
dtype: float64


In [57]:
# Step 6: Map transaction types into UPI-like categories
type_map = {
    'TRANSFER': 'upi_transfer',   # P2P transfer
    'CASH-OUT': 'upi_transfer',   # behaves like transfer (user → cash agent)
    'PAYMENT': 'upi_payment',     # user → merchant
    'DEBIT': 'other',             # less relevant
    'CASH-IN': 'other'            # cash deposited into account
}

df['upi_type'] = df['type'].map(type_map).fillna('other').astype('category')

# ✅ quick sanity check
print("Original types distribution:\n", df['type'].value_counts(normalize=True))
print("\nMapped UPI types distribution:\n", df['upi_type'].value_counts(normalize=True))
print("\nMissing values in upi_type:", df['upi_type'].isnull().sum())


Original types distribution:
 type
CASH_OUT    0.351663
PAYMENT     0.338146
CASH_IN     0.219923
TRANSFER    0.083756
DEBIT       0.006512
Name: proportion, dtype: float64

Mapped UPI types distribution:
 upi_type
other           0.578098
upi_payment     0.338146
upi_transfer    0.083756
Name: proportion, dtype: float64

Missing values in upi_type: 0


In [58]:
df[['type', 'upi_type']].head(10) 

,type,upi_type
0,PAYMENT,upi_payment
1,PAYMENT,upi_payment
2,TRANSFER,upi_transfer
3,CASH_OUT,other
4,PAYMENT,upi_payment
5,PAYMENT,upi_payment
6,PAYMENT,upi_payment
7,PAYMENT,upi_payment
8,PAYMENT,upi_payment
9,DEBIT,other


In [59]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud', 'datetime', 'hour', 'dayofweek', 'is_weekend',
       'errorBalanceOrig', 'errorBalanceDest', 'has_balance_mismatch',
       'upi_type', 'orig_amount_std'],
      dtype='object')

In [60]:
missing_frac = df.isnull().mean().sort_values(ascending=False)
print("Columns with >0 missing fraction:\n", missing_frac[missing_frac > 0])

Columns with >0 missing fraction:
 orig_amount_std    0.997075
dtype: float64


Dropping / Deleting the missing vakues columns as it is not required.

In [61]:
if 'orig_amount_std' in df.columns:
    df = df.drop(columns=['orig_amount_std'])

In [62]:
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'datetime', 'hour', 'dayofweek', 'is_weekend', 'errorBalanceOrig', 'errorBalanceDest', 'has_balance_mismatch', 'upi_type']


In [63]:
missing_frac = df.isnull().mean().sort_values(ascending=False)
print("Columns with >0 missing fraction:\n", missing_frac[missing_frac > 0])

Columns with >0 missing fraction:
 Series([], dtype: float64)


Coverting the upi categorical data into numerical data

In [64]:
df = pd.get_dummies(df, columns=['upi_type'], drop_first=True)

print("New columns after encoding:\n", df.columns.tolist())

New columns after encoding:
 ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'datetime', 'hour', 'dayofweek', 'is_weekend', 'errorBalanceOrig', 'errorBalanceDest', 'has_balance_mismatch', 'upi_type_upi_payment', 'upi_type_upi_transfer']


In [65]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,datetime,hour,dayofweek,is_weekend,errorBalanceOrig,errorBalanceDest,has_balance_mismatch,upi_type_upi_payment,upi_type_upi_transfer
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,2017-01-01 01:00:00,1,6,1,0.0,9839.64,1,True,False
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,2017-01-01 01:00:00,1,6,1,0.0,1864.28,1,True,False
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,2017-01-01 01:00:00,1,6,1,0.0,181.00,1,False,True
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,2017-01-01 01:00:00,1,6,1,0.0,21363.00,1,False,False
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,2017-01-01 01:00:00,1,6,1,0.0,11668.14,1,True,False


Final Dataset cleaning step by deleting all the unnecessary columns and values reducing noise.

In [66]:
df = df.drop(columns=[        
    'type',     
    'nameDest',     
    'isFlaggedFraud'
])

print("Final cleaned dataset shape:", df.shape)
print("Remaining columns:", df.columns.tolist())


Final cleaned dataset shape: (6362620, 17)
Remaining columns: ['step', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'datetime', 'hour', 'dayofweek', 'is_weekend', 'errorBalanceOrig', 'errorBalanceDest', 'has_balance_mismatch', 'upi_type_upi_payment', 'upi_type_upi_transfer']


In [75]:
print("Total rows:", len(df))

print("\nChecking nameorig:")
print("Is nameorig present?", 'nameorig' in df.columns)
print("Unique nameorig count:", df['nameorig'].nunique() if 'nameorig' in df.columns else "N/A")

print("\nTransactions per user (top 5):")
if 'nameorig' in df.columns:
    print(df['nameorig'].value_counts().head())

Total rows: 6362620

Checking nameorig:
Is nameorig present? False
Unique nameorig count: N/A

Transactions per user (top 5):


Saving the dataset

In [67]:
df.to_csv("cleaned_paysim_lstm.csv", index=False)

In [73]:
df.tail()

,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,datetime,hour,dayofweek,is_weekend,errorBalanceOrig,errorBalanceDest,has_balance_mismatch,upi_type_upi_payment,upi_type_upi_transfer
6362615,743,339682.13,C786484425,339682.13,0.0,0.00,339682.13,1,2017-01-31 23:00:00,23,1,0,0.0,0.000000e+00,0,False,False
6362616,743,6311409.28,C1529008245,6311409.28,0.0,0.00,0.00,1,2017-01-31 23:00:00,23,1,0,0.0,6.311409e+06,1,False,True
6362617,743,6311409.28,C1162922333,6311409.28,0.0,68488.84,6379898.11,1,2017-01-31 23:00:00,23,1,0,0.0,1.000000e-02,1,False,False
6362618,743,850002.52,C1685995037,850002.52,0.0,0.00,0.00,1,2017-01-31 23:00:00,23,1,0,0.0,8.500025e+05,1,False,True
6362619,743,850002.52,C1280323807,850002.52,0.0,6510099.11,7360101.63,1,2017-01-31 23:00:00,23,1,0,0.0,9.313226e-10,0,False,False


In [72]:
df.columns.tolist()

['step',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'datetime',
 'hour',
 'dayofweek',
 'is_weekend',
 'errorBalanceOrig',
 'errorBalanceDest',
 'has_balance_mismatch',
 'upi_type_upi_payment',
 'upi_type_upi_transfer']

In [74]:
print("Total rows:", len(df))

print("\nChecking nameorig:")
print("Is nameorig present?", 'nameorig' in df.columns)
print("Unique nameorig count:", df['nameorig'].nunique() if 'nameorig' in df.columns else "N/A")

print("\nTransactions per user (top 5):")
if 'nameorig' in df.columns:
    print(df['nameorig'].value_counts().head())

Total rows: 6362620

Checking nameorig:
Is nameorig present? False
Unique nameorig count: N/A

Transactions per user (top 5):
